# JN0d · What a pandas DataFrame is

*On-ramp 4 of 8.*

We have thirty thousand permits in one table. How do we pull out *exactly* the one we want — the building at 2352 Shattuck — without scrolling? And how do we ask the table to summarise itself? With a **DataFrame**.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0c · What a function is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0c_function.ipynb)  |  Next: [JN0e · One set of numbers, many pictures](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0e_charts.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


In [2]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, the permit feed, and the project's shared code. The two knobs near the top are all a student changes to run another city.

In [3]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


repo root : /Users/johngage/berkeley-data
feed files: ['BP_Annual Permit Report-2023-2025.xlsx', 'BP_Annual Permit Report-2018-2022.xlsx']


## A table you query, not a sheet you scroll

A **DataFrame** (from the **pandas** library) is a table the computer can reason about: **rows** (one per permit) and named **columns**, with an **index** to address rows. Unlike a spreadsheet you scroll by eye, you ask it questions in code — and the answer is itself a table you can keep working on.

In [4]:
import pandas as pd, glob
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)   # stack every yearly file into one table
df = df[df['PermitNumber'].notna()].copy()        # drop rows that have no permit number
df['units_n'] = pd.to_numeric(df['NumberUnits'], errors='coerce')   # add a numeric units column (bad values -> NaN)
df['year'] = df['PermitNumber'].str.extract(r'^[A-Za-z]+(\d{4})')[0]   # add a year column pulled from the permit number
print(f'{len(df):,} permits loaded, columns ready')

32,202 permits loaded, columns ready


In [5]:
print('shape (rows, columns):', df.shape)          # how big the table is
print('a few columns:', list(df.columns)[:8])      # the first handful of column names
df.head(3)                                          # the first few rows

shape (rows, columns): (32202, 28)
a few columns: ['PermitNumber', 'Submittal Date', 'Issuance Status', 'Unnamed: 3', 'Issuance Date', 'Finaled Status', 'Finaled Date', 'Completed']


,PermitNumber,Submittal Date,Issuance Status,Unnamed: 3,Issuance Date,Finaled Status,Finaled Date,Completed,Completed Date,Parcel Number,...,Detached,Work Type,OccType,SubType,NumberUnits,UnitsAdded,UnitsRemoved,CO Required,units_n,year
0,B2004-04914,2004-11-08 00:00:00,Issued,NaN,06/25/2024,NaN,NaN,NaN,NaN,060 241605100,...,NaN,Addition,97R3,Residential,NaN,NaN,NaN,NaN,NaN,2004
1,B2005-01341,2005-04-04 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,059 225501900,...,NaN,New,97R3,Residential,NaN,NaN,NaN,NaN,NaN,2005
2,B2006-03740,2006-09-15 00:00:00,NaN,NaN,NaN,Finaled,2024-12-05 00:00:00,NaN,NaN,063 295604100,...,NaN,Addition,97R3,Residential,NaN,1.00,NaN,NaN,NaN,2006


## Pull one row out of thirty thousand

To find specific rows you write a **condition** — a *boolean mask*, a column of True/False — and keep the rows where it's True. Here's the single permit for the new building at 2352 Shattuck:

In [6]:
hit = df[df['PermitNumber'] == 'B2019-05574']      # keep only the row(s) matching this permit number
hit[['PermitNumber','StreetNumber','StreetName','Work Type','NumberUnits','Finaled Date']]   # show selected columns

,PermitNumber,StreetNumber,StreetName,Work Type,NumberUnits,Finaled Date
23701,B2019-05574,2352,SHATTUCK,New,135,2022-01-14 00:00:00


## Make the table summarise itself

Three everyday moves do most of the work: **count** the values in a column, **sort** to find extremes, and **group** to aggregate. First, what *kinds* of permit are there?

In [7]:
df['Work Type'].value_counts()   # count how many permits fall under each work type

Work Type
Alteration             27076
New                     1773
Addition/Alteration     1729
Addition                 556
Demolition               373
Sign                      63
Name: count, dtype: int64

**Sort** to surface the biggest projects — the largest unit counts float to the top:

In [8]:
# sort by unit count, biggest first, and show the top rows
(df.sort_values('units_n', ascending=False)
   [['PermitNumber','StreetNumber','StreetName','Work Type','units_n']]
   .head())

,PermitNumber,StreetNumber,StreetName,Work Type,units_n
8057,B2024-01924-REV08,1598,UNIVERSITY,New,207.0
8051,B2024-01924-DEF03,1598,UNIVERSITY,New,207.0
8060,B2024-01924-REV11,1598,UNIVERSITY,New,207.0
8059,B2024-01924-REV10,1598,UNIVERSITY,New,207.0
8048,B2024-01924,1598,UNIVERSITY,New,207.0


In [9]:
_top = df.sort_values('units_n', ascending=False).iloc[0]   # the single biggest permit by units
md(f'''**Sorting is discovery.** The single largest permit in the feed is **{int(_top['units_n'])} units** at {_top['StreetNumber']} {_top['StreetName']} (`{_top['PermitNumber']}`). You didn't go looking for it — you asked the table to order itself and the biggest buildings announced themselves.''')

**Sorting is discovery.** The single largest permit in the feed is **207 units** at 1598 UNIVERSITY (`B2024-01924-REV08`). You didn't go looking for it — you asked the table to order itself and the biggest buildings announced themselves.

**Group** to aggregate — total units by work type, the workhorse of every later analysis:

In [10]:
# group by work type and total the units in each group, largest first
by_type = df.groupby('Work Type')['units_n'].sum().sort_values(ascending=False)
by_type.head(6)

Work Type
Alteration             43573.0
New                    40876.0
Addition/Alteration     3250.0
Addition                 565.0
Demolition               130.0
Sign                      46.0
Name: units_n, dtype: float64

**Next — JN0e:** we can pull numbers out and summarise them — now how do we *show* them so they make an argument? One set of numbers, many pictures.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0c · What a function is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0c_function.ipynb)  |  Next: [JN0e · One set of numbers, many pictures](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0e_charts.ipynb) →